In [0]:
%pip install -U dspy-ai databricks-dspy mlflow databricks-sdk sentence-transformers scikit-learn

# Models and Agent Definition

In [0]:
import dspy
import databricks_dspy
from databricks.sdk import WorkspaceClient

# ==========================================
# 1. Setup the LLMs (Abstracted for A/B Testing)
# ==========================================
w = WorkspaceClient()

# Define Model 1 (Primary / Heavyweight)
model1 = databricks_dspy.DatabricksLM(
    model="databricks/databricks-meta-llama-3-3-70b-instruct",
    workspace_client=w
)

# Define Model 2 (Baseline / Comparison)
model2 = databricks_dspy.DatabricksLM(
    model="databricks/databricks-qwen3-next-80b-a3b-instruct",
    workspace_client=w
)

# Set Model 1 as the default agent for now. 
# (To test Model 2 later, simply change this to lm=model2)
dspy.settings.configure(lm=model1)
print("Successfully connected to Databricks Model Serving!")

# ==========================================
# 2. Define DSPy Signatures & Agent
# ==========================================
class VetTrackClassifier(dspy.Signature):
    """
    You are a triage agent for VetTrack (a Veterinary CRM). 
    Classify the incoming ticket. If the ticket is NOT related to IT, VetTrack, or Veterinary software, you must reject it.
    """
    ticket_description = dspy.InputField(desc="The raw customer support message.")
    is_relevant = dspy.OutputField(desc="Boolean 'True' if related to VetTrack/IT, 'False' if irrelevant/spam.")
    rejection_message = dspy.OutputField(desc="If is_relevant is False, politely explain that you can only help with VetTrack CRM issues. If True, output 'N/A'.")
    ticket_subject = dspy.OutputField(desc="The category of the issue (e.g., Login, Billing, Scheduling) if relevant.")
    priority = dspy.OutputField(desc="'Low', 'Medium', 'High', or 'Critical' based on urgency.")

class VetTrackResolver(dspy.Signature):
    """Draft a professional IT resolution for a VetTrack customer based on past successful resolutions."""
    ticket_subject = dspy.InputField(desc="The classified subject of the issue.")
    ticket_description = dspy.InputField(desc="The user's problem.")
    retrieved_context = dspy.InputField(desc="Past resolved tickets from the vector database.")
    draft_resolution = dspy.OutputField(desc="A polite, detailed, and professional email resolving the user's issue.")

class VetTrackAgent(dspy.Module):
    def __init__(self, retriever_function):
        super().__init__()
        self.classifier = dspy.Predict(VetTrackClassifier)
        self.resolver = dspy.Predict(VetTrackResolver)
        self.retriever_function = retriever_function

    def forward(self, ticket_description):
        classification = self.classifier(ticket_description=ticket_description)
        
        # Graceful Rejection
        if classification.is_relevant == "False":
            return dspy.Prediction(
                resolution=classification.rejection_message, priority="N/A",
                escalated=False, status="Rejected"
            )
            
        # Escalation Logic
        escalated = True if classification.priority in ["High", "Critical"] else False
        
        # RAG Retrieval
        past_tickets_context = self.retriever_function(ticket_description)
        
        # Resolution Draft
        resolution = self.resolver(
            ticket_subject=classification.ticket_subject,
            ticket_description=ticket_description,
            retrieved_context=past_tickets_context
        )
        
        return dspy.Prediction(
            resolution=resolution.draft_resolution, priority=classification.priority,
            escalated=escalated, status="Resolved"
        )

# Data Splitting & Formatting

In [0]:
import pandas as pd
import numpy as np
import dspy

# 1. Pull data from Unity Catalog and filter for Closed tickets only
# Note: We convert to Pandas here because DSPy and scikit-learn work natively with Pandas/Numpy
df_closed_spark = spark.table("main.default.final_project").filter("Ticket_Status == 'Closed'")
df_closed = df_closed_spark.toPandas()

# 2. Shuffle the dataset to ensure a random distribution of ticket types
df_closed = df_closed.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total Closed Tickets available: {len(df_closed)}")

# 3. Carve out the splits!
# - First 50 rows for DSPy Prompt Optimization (Training)
# - Next 50 rows for MLflow Final Grading (Evals)
# - Everything else (~2,600+ rows) goes into the Vector Knowledge Base (RAG)
df_train = df_closed.iloc[:50].copy()
df_eval = df_closed.iloc[50:100].copy()
df_kb = df_closed.iloc[100:].copy()

print(f"Allocated -> Training: {len(df_train)} | Evals: {len(df_eval)} | Knowledge Base: {len(df_kb)}")

# 4. Convert our Pandas rows into DSPy 'Example' objects
# DSPy requires data to be in this specific format to run evaluations and training.
# We explicitly tell DSPy that 'ticket_description' is the input, and the rest are the expected answers.
def create_dspy_examples(df):
    examples = []
    for _, row in df.iterrows():
        ex = dspy.Example(
            ticket_description=row['Ticket_Description'],
            resolution=row['Resolution'],
            priority=row['Ticket_Priority']
        ).with_inputs('ticket_description')
        examples.append(ex)
    return examples

trainset = create_dspy_examples(df_train)
evalset = create_dspy_examples(df_eval)

# Build the Vector Retriever (RAG)

In [0]:
# Make sure you have these installed: %pip install sentence-transformers scikit-learn
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the exact same embedding model you used in your EDA notebook
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Extract the pre-calculated Knowledge Base embeddings into a fast numpy array
# (This acts as our localized Vector Database)
kb_embeddings = np.vstack(df_kb['embeddings'].values)

# 3. The Real Retriever Function
def real_vector_retriever(query, k=3):
    """
    Takes a user's IT problem, embeds it, and finds the top 'k' most 
    similar historical tickets and their resolutions from the Knowledge Base.
    """
    # Embed the incoming user query
    query_emb = embed_model.encode([query])
    
    # Calculate Cosine Similarity between the query and the entire Knowledge Base
    similarities = cosine_similarity(query_emb, kb_embeddings)[0]
    
    # Get the indices of the top K highest similarity scores
    top_indices = np.argsort(similarities)[-k:][::-1]
    
    # Build the context string to feed to the LLM
    retrieved_context = []
    for idx in top_indices:
        row = df_kb.iloc[idx]
        retrieved_context.append(
            f"[PAST TICKET]\n"
            f"Problem: {row['Ticket_Description']}\n"
            f"Subject: {row['Ticket_Subject']}\n"
            f"Resolution: {row['Resolution']}"
        )
    
    # Join them together into one string
    return "\n\n".join(retrieved_context)

# Test the RAG Agent

In [0]:
# Re-instantiate the agent with our real Databricks data retriever!
production_agent = VetTrackAgent(retriever_function=real_vector_retriever)

# Let's test it by pulling a random ticket from our Evaluation Set!
test_ticket = evalset[0].ticket_description
true_resolution = evalset[0].resolution

print("=== REAL USER QUERY ===")
print(test_ticket)

print("\n=== AGENT'S GENERATED RESPONSE ===")
result = production_agent(ticket_description=test_ticket)
print(f"Priority Assigned: {result.priority}")
print(f"Escalated: {result.escalated}")
print(f"Resolution:\n{result.resolution}")

print("\n=== ACTUAL HISTORICAL RESOLUTION (For comparison) ===")
print(true_resolution)